#### Download models

In [ ]:
!mkdir models
!wget --header="Authorization: Bearer <HF_TOKEN>" https://huggingface.co/wkentaro/sam3-onnx-models/resolve/main/sam3_decoder.onnx
!wget --header="Authorization: Bearer <HF_TOKEN>" https://huggingface.co/wkentaro/sam3-onnx-models/resolve/main/sam3_language_encoder.onnx
!wget --header="Authorization: Bearer <HF_TOKEN>" https://huggingface.co/wkentaro/sam3-onnx-models/resolve/main/sam3_image_encoder.onnx

In [29]:
import pathlib
import typing
import cv2
import imgviz
import numpy as np
import onnxruntime
import PIL.Image
from loguru import logger
from numpy.typing import NDArray

from osam._models.yoloworld.clip import tokenize

In [30]:
sess_image = onnxruntime.InferenceSession(
    "models/sam3_image_encoder.onnx"
)
sess_language = onnxruntime.InferenceSession(
    "models/sam3_language_encoder.onnx"
)
sess_decode = onnxruntime.InferenceSession(
    "models/sam3_decoder.onnx"
)

print("Models loaded successfully.")

Models loaded successfully.


In [31]:
# Path to image
IMAGE_PATH = pathlib.Path("demo2.jpeg")

# Choose ONE:
TEXT_PROMPT = "person"   # set to None if using box prompt
BOX_PROMPT = None        # format: [cx, cy, w, h] (normalized)

In [32]:
image: PIL.Image.Image = PIL.Image.open(IMAGE_PATH).convert("RGB")
print("Original image size:", image.size)

# Resize and convert to CHW format
image_array = np.asarray(image.resize((1008, 1008)))
image_tensor = image_array.transpose(2, 0, 1)

Original image size: (2048, 1365)


In [33]:
%%time
output = sess_image.run(
    None,
    {"image": image_tensor}
)

vision_pos_enc = output[:3]
backbone_fpn = output[3:]

print("Image encoder output ready.")

Image encoder output ready.
CPU times: total: 7min 28s
Wall time: 52 s


In [34]:
text_prompt = TEXT_PROMPT if TEXT_PROMPT else "visual"

tokens = tokenize(
    texts=[text_prompt],
    context_length=32
)

output = sess_language.run(
    None,
    {"tokens": tokens}
)

language_mask = output[0]
language_features = output[1]

print("Language encoder output ready.")

Language encoder output ready.


In [35]:
if BOX_PROMPT is None:
    box_coords = np.array([[0, 0, 0, 0]], dtype=np.float32).reshape(1, 1, 4)
    box_labels = np.array([[1]], dtype=np.int64)
    box_masks = np.array([True], dtype=np.bool_).reshape(1, 1)
else:
    box_coords = np.array(BOX_PROMPT, dtype=np.float32).reshape(1, 1, 4)
    box_labels = np.array([[1]], dtype=np.int64)
    box_masks = np.array([False], dtype=np.bool_).reshape(1, 1)

In [36]:
output = sess_decode.run(
    None,
    {
        "original_height": np.array(image.height, dtype=np.int64),
        "original_width": np.array(image.width, dtype=np.int64),
        "backbone_fpn_0": backbone_fpn[0],
        "backbone_fpn_1": backbone_fpn[1],
        "backbone_fpn_2": backbone_fpn[2],
        "vision_pos_enc_2": vision_pos_enc[2],
        "language_mask": language_mask,
        "language_features": language_features,
        "box_coords": box_coords,
        "box_labels": box_labels,
        "box_masks": box_masks,
    }
)

boxes, scores, masks = output
print("Decoder finished.")


Decoder finished.


In [37]:
viz = imgviz.instances2rgb(
    image=np.asarray(image),
    masks=masks[:, 0, :, :],
    bboxes=boxes[:, [1, 0, 3, 2]],
    labels=np.arange(len(masks)) + 1,
    captions=[f"{text_prompt}: {s:.0%}" for s in scores],
    font_size=max(1, min(image.size) // 40),
)

imgviz.io.pil_imshow(viz)

### **Inference - Based on input type**

In [ ]:
# COMMON SETUP (RUN ONCE)
import pathlib
import numpy as np
import onnxruntime
import PIL.Image
from osam._models.yoloworld.clip import tokenize
import time

sess_image = onnxruntime.InferenceSession("models/sam3_image_encoder.onnx")
sess_language = onnxruntime.InferenceSession("models/sam3_language_encoder.onnx")
sess_decode = onnxruntime.InferenceSession("models/sam3_decoder.onnx")

IMAGE_PATH = pathlib.Path("demo.jpg")

image = PIL.Image.open(IMAGE_PATH).convert("RGB")
image_tensor = np.asarray(image.resize((1008, 1008))).transpose(2, 0, 1)

vision_out = sess_image.run(None, {"image": image_tensor})
vision_pos_enc = vision_out[:3]
backbone_fpn = vision_out[3:]

#### **1. PASS ONLY BOUNDING BOX**

In [ ]:
# Normalized box: cx, cy, w, h
BOX_PROMPT = [0.5, 0.5, 0.4, 0.4]

box_coords = np.array(BOX_PROMPT, dtype=np.float32).reshape(1, 1, 4)
box_labels = np.array([[1]], dtype=np.int64)
box_masks = np.array([False], dtype=np.bool_).reshape(1, 1)

# Dummy language input
tokens = tokenize(["visual"], context_length=32)
lang_out = sess_language.run(None, {"tokens": tokens})
language_mask, language_features = lang_out[0], lang_out[1]

st = time.perf_counter()
boxes, scores, masks = sess_decode.run(
    None,
    {
        "original_height": np.array(image.height, dtype=np.int64),
        "original_width": np.array(image.width, dtype=np.int64),
        "backbone_fpn_0": backbone_fpn[0],
        "backbone_fpn_1": backbone_fpn[1],
        "backbone_fpn_2": backbone_fpn[2],
        "vision_pos_enc_2": vision_pos_enc[2],
        "language_mask": language_mask,
        "language_features": language_features,
        "box_coords": box_coords,
        "box_labels": box_labels,
        "box_masks": box_masks,
    }
)
en = time.perf_counter()
print("Time taken: {:.2f}".format(en - st))

#### **2. PASS ONLY POINT**

In [ ]:
# (Point encoded as tiny box)

# Normalized point (px, py)
POINT = [0.6, 0.4]
EPS = 0.001  # tiny box size

box_coords = np.array(
    [POINT[0], POINT[1], EPS, EPS], dtype=np.float32
).reshape(1, 1, 4)

box_labels = np.array([[1]], dtype=np.int64)
box_masks = np.array([False], dtype=np.bool_).reshape(1, 1)

tokens = tokenize(["visual"], context_length=32)
lang_out = sess_language.run(None, {"tokens": tokens})
language_mask, language_features = lang_out[0], lang_out[1]

st = time.perf_counter()
boxes, scores, masks = sess_decode.run(
    None,
    {
        "original_height": np.array(image.height, dtype=np.int64),
        "original_width": np.array(image.width, dtype=np.int64),
        "backbone_fpn_0": backbone_fpn[0],
        "backbone_fpn_1": backbone_fpn[1],
        "backbone_fpn_2": backbone_fpn[2],
        "vision_pos_enc_2": vision_pos_enc[2],
        "language_mask": language_mask,
        "language_features": language_features,
        "box_coords": box_coords,
        "box_labels": box_labels,
        "box_masks": box_masks,
    }
)

en = time.perf_counter()
print("Time taken: {:.2f}".format(en - st))

#### **3. PASS ONLY TEXT**

In [ ]:
TEXT_PROMPT = "Camera"

box_coords = np.array([[0, 0, 0, 0]], dtype=np.float32).reshape(1, 1, 4)
box_labels = np.array([[1]], dtype=np.int64)
box_masks = np.array([True], dtype=np.bool_).reshape(1, 1)

tokens = tokenize([TEXT_PROMPT], context_length=32)
lang_out = sess_language.run(None, {"tokens": tokens})
language_mask, language_features = lang_out[0], lang_out[1]


st = time.perf_counter()
boxes, scores, masks = sess_decode.run(
    None,
    {
        "original_height": np.array(image.height, dtype=np.int64),
        "original_width": np.array(image.width, dtype=np.int64),
        "backbone_fpn_0": backbone_fpn[0],
        "backbone_fpn_1": backbone_fpn[1],
        "backbone_fpn_2": backbone_fpn[2],
        "vision_pos_enc_2": vision_pos_enc[2],
        "language_mask": language_mask,
        "language_features": language_features,
        "box_coords": box_coords,
        "box_labels": box_labels,
        "box_masks": box_masks,
    }
)

en = time.perf_counter()
print("Time taken: {:.2f}".format(en - st))

In [ ]:
viz = imgviz.instances2rgb(
    image=np.asarray(image),
    masks=masks[:, 0, :, :],
    bboxes=boxes[:, [1, 0, 3, 2]],
    labels=np.arange(len(masks)) + 1,
    captions=[f"{text_prompt}: {s:.0%}" for s in scores],
    font_size=max(1, min(image.size) // 40),
)
imgviz.io.pil_imshow(viz)

#### **Local Endpoint**

In [ ]:
import requests

url = "http://127.0.0.1:8000/segment"

with open("demo.jpg", "rb") as f:
    response = requests.post(
        url,
        files={"file": f},
        data={"text_prompt": "person"}
    )

print("Status:", response.status_code)

if response.headers.get("content-type", "").startswith("application/json"):
    print(response.json())
else:
    print("Non-JSON response:")
    print(response.text)

In [ ]:
print(response.json())